In [1]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_stats as graph_stats
import utils.graph_utils as graph_utils
import utils.vis as vis

from config_handlers.rk_handler import RKHandler

# Main

In [2]:
# Load RK config
rk_hdl = RKHandler()
MODELS = rk_hdl.get_model_list()
rk_hdl.set_curr_model(MODELS[0])
DATASETS = rk_hdl.get_ds_list()
base_dir = Path("..")

all_datasets = DATASETS * 4
all_models = MODELS * 4
all_models.sort()

## Count valid RK entries

In [3]:
def get_valid_rks(all_models: list[str], all_datasets: list[str], version: int) -> dict:
    all_rks = {}

    for model, ds in zip(all_models, all_datasets):
        rk_hdl.set_curr_model(model)
        rk_hdl.set_curr_ds(ds)
        parsed_rk_path = base_dir.joinpath(
            str(rk_hdl.get_parsed_rk_path()).format(version)
        )
        parsed_rks = data_io.load_jsonl(parsed_rk_path)

        # Set up dict for current model x dataset combination
        if model not in all_rks:
            all_rks[model] = {}
        if ds not in all_rks[model]:
            all_rks[model][ds] = {}
        rk_dict = {}

        # Create entry for each sample
        for curr_sample in parsed_rks:
            question_id = curr_sample["question_id"]
            # Skip if there is any structured triple
            if len(curr_sample["triple_objs"]) == 0:
                # print(f"No RK found for {question_id} ({model} x {ds})")
                continue

            # Otherwise, record entry
            if question_id not in rk_dict:
                rk_dict[question_id] = curr_sample["triple_objs"]

        # Save for later
        all_rks[model][ds] = rk_dict

    return all_rks

In [4]:
def print_rk_details(data_det: dict, data_sam: dict) -> None:
    for model in MODELS:
        print("=" * 10, f"{model.upper()}", "=" * 10)
        for ds in DATASETS:
            print("-" * 50)
            print(f"{ds.upper()}")
            print("-" * 50)
            print("\t\t = Greedy = \t = Nucleus =")

            # Get the number of entries
            n_samples_det = len(data_det["sample_stats"][model][ds])
            n_samples_sam = len(data_sam["sample_stats"][model][ds])
            print(f"Entries \t {n_samples_det} \t\t {n_samples_sam}")

            # Get the number of concepts extracted
            n_concepts_det = data_det["summaries"][model][ds]["count_concepts"]
            n_concepts_sam = data_sam["summaries"][model][ds]["count_concepts"]
            print(f"N. Nodes \t {n_concepts_det} \t\t {n_concepts_sam}")

            # Get the number of relations extracted
            n_rels_det = data_det["summaries"][model][ds]["count_preds"]
            n_sam_det = data_sam["summaries"][model][ds]["count_preds"]
            print(f"N. RKs \t\t {n_rels_det} \t\t {n_sam_det}")

            # Compare IC and Entropy at the graph level
            avg_ic_det = data_det["summaries"][model][ds]["avg_ic_graph"]
            std_ic_det = data_det["summaries"][model][ds]["std_ic_graph"]
            avg_ic_sam = data_sam["summaries"][model][ds]["avg_ic_graph"]
            std_ic_sam = data_sam["summaries"][model][ds]["std_ic_graph"]

            avg_h_det = data_det["summaries"][model][ds]["avg_h_graph"]
            std_h_det = data_det["summaries"][model][ds]["std_h_graph"]
            avg_h_sam = data_sam["summaries"][model][ds]["avg_h_graph"]
            std_h_sam = data_sam["summaries"][model][ds]["std_h_graph"]

            print(
                f"IC \t\t {avg_ic_det:.2f} ({std_ic_det:.2f}) \t {avg_ic_sam:.2f} ({std_ic_sam:.2f})"
            )
            print(
                f"H \t\t {avg_h_det:.2f} ({std_h_det:.2f}) \t {avg_h_sam:.2f} ({std_h_sam:.2f})"
            )

Loading dictionary from VG1800 with concept, predicates, and their counts and frequencies

In [5]:
# Loading concept and predicate frequencies from VG1800 dataset from IETrans.
vg1800_dict = graph_utils.load_vg1800_dict()

In [6]:
def get_rk_details(rks: dict) -> dict:
    data = {
        "graphs": {model: {} for model in MODELS},
        "summaries": {model: {} for model in MODELS},
        "sample_stats": {model: {} for model in MODELS},
        "count_concepts": {model: {} for model in MODELS},
        "count_preds": {model: {} for model in MODELS},
    }

    for model, ds in zip(all_models, all_datasets):
        print(f"Processing {model} -- {ds}")
        rk_hdl.set_curr_model(model)
        rk_hdl.set_curr_ds(ds)
        if ds not in data["graphs"][model]:
            data["graphs"][model][ds] = {}

        # Create NX graphs for each data sample
        for q_idx, q_data in rks[model][ds].items():
            data["graphs"][model][ds][q_idx] = graph_utils.rk_to_nx(q_data)

        sample_stats, stats_summary = graph_stats.compute_stats(
            data["graphs"][model][ds], vg1800_dict
        )

        # Update
        data["sample_stats"][model][ds] = sample_stats
        data["summaries"][model][ds] = stats_summary
        data["count_concepts"][model][ds] = stats_summary["count_concepts"]
        data["count_preds"][model][ds] = stats_summary["count_preds"]

    return data

Greedy decoding (v4)

In [7]:
prompt_version = 4

# Load RK data
rks_det = get_valid_rks(all_models, all_datasets, prompt_version)
# Process RK data
data_det = get_rk_details(rks_det)

Processing internvl2 -- llava-bench
Processing internvl2 -- mmbench
Processing internvl2 -- seed
Processing internvl2 -- vqav2
Processing llava-1.6 -- llava-bench
Processing llava-1.6 -- mmbench
Processing llava-1.6 -- seed
Processing llava-1.6 -- vqav2
Processing minigpt4 -- llava-bench
Processing minigpt4 -- mmbench
Processing minigpt4 -- seed
Processing minigpt4 -- vqav2
Processing sharegpt4v -- llava-bench
Processing sharegpt4v -- mmbench
Processing sharegpt4v -- seed
Processing sharegpt4v -- vqav2


Nucleus sampling (v5)

In [8]:
prompt_version = 5

# Load RK data
rks_sam = get_valid_rks(all_models, all_datasets, prompt_version)
# Process RK data
data_sam = get_rk_details(rks_sam)

Processing internvl2 -- llava-bench
Processing internvl2 -- mmbench
Processing internvl2 -- seed
Processing internvl2 -- vqav2
Processing llava-1.6 -- llava-bench
Processing llava-1.6 -- mmbench
Processing llava-1.6 -- seed
Processing llava-1.6 -- vqav2
Processing minigpt4 -- llava-bench
Processing minigpt4 -- mmbench
Processing minigpt4 -- seed
Processing minigpt4 -- vqav2
Processing sharegpt4v -- llava-bench
Processing sharegpt4v -- mmbench
Processing sharegpt4v -- seed
Processing sharegpt4v -- vqav2


Comparison

In [9]:
print_rk_details(data_det, data_sam)

========== INTERNVL2 ==========
--------------------------------------------------
LLAVA-BENCH
--------------------------------------------------
		 = Greedy = 	 = Nucleus =
Entries 	 58 		 58
N. Nodes 	 634 		 628
N. RKs 		 565 		 584
IC 		 17.63 (2.12) 	 17.89 (2.45)
H 		 0.04 (0.02) 	 0.04 (0.02)
--------------------------------------------------
MMBENCH
--------------------------------------------------
		 = Greedy = 	 = Nucleus =
Entries 	 139 		 139
N. Nodes 	 1272 		 1298
N. RKs 		 1287 		 1346
IC 		 15.31 (2.38) 	 15.43 (2.38)
H 		 0.05 (0.03) 	 0.05 (0.02)
--------------------------------------------------
SEED
--------------------------------------------------
		 = Greedy = 	 = Nucleus =
Entries 	 150 		 150
N. Nodes 	 1219 		 1230
N. RKs 		 1254 		 1283
IC 		 16.33 (2.62) 	 16.37 (2.51)
H 		 0.04 (0.02) 	 0.04 (0.02)
--------------------------------------------------
VQAV2
--------------------------------------------------
		 = Greedy = 	 = Nucleus =
Entries 	 150 		 150
N. 

# Save results to disk

Save statistics

In [ ]:
# Create output dir for statistics
out_dir_det = Path("..", "data", "stats", "rk_det")
data_io.make_dir(out_dir_det)
out_dir_sam = Path("..", "data", "stats", "rk_sam")
data_io.make_dir(out_dir_sam)

for model, ds in zip(all_models, all_datasets):
    print(f"Saving {model} -- {ds}")

    # Save statistics greedy decoding
    ## Summaries
    summary_det_file = out_dir_det.joinpath(model, ds, "summary.json")
    data_io.make_dir(summary_det_file.parent)
    data_io.save_json(data_det["summaries"][model][ds], summary_det_file)
    ## Sample-level
    sample_def_file = out_dir_det.joinpath(model, ds, "samples.json")
    data_io.save_json(data_det["sample_stats"][model][ds], sample_def_file)

    # Save statistics nucleus sampling decoding
    ## Summaries
    summary_sam_file = out_dir_sam.joinpath(model, ds, "summary.json")
    data_io.make_dir(summary_sam_file.parent)
    data_io.save_json(data_sam["summaries"][model][ds], summary_sam_file)
    # Sample-level
    sample_sam_file = out_dir_det.joinpath(model, ds, "samples.json")
    data_io.save_json(data_sam["sample_stats"][model][ds], sample_sam_file)

Save pickled graphs

In [ ]:
# Create output dir for pkl graphs
pkl_dir = Path("..", "data", "really_know", "parsed_pkl")
data_io.make_dir(pkl_dir)

for model, ds in zip(all_models, all_datasets):
    print(f"Saving {model} -- {ds}")

    # Save pickled graphs
    pkl_out_file = pkl_dir.joinpath(model, ds, "rk_nx.pkl")
    data_io.make_dir(pkl_out_file.parent)
    data_io.save_pickle(data_det["graphs"][model][ds], pkl_out_file)

Save statistics graphs

In [ ]:
# Create output dir for charts
charts_dir = Path(".", "stats_charts")
data_io.make_dir(charts_dir)

# Greedy decoding
vis.plot_boxplot_rk(
    data_det["count_concepts"],
    data_det["count_preds"],
    charts_dir.joinpath("box_rk_greedy.pdf"),
)

# Nucleus sampling
vis.plot_boxplot_rk(
    data_sam["count_concepts"],
    data_sam["count_preds"],
    charts_dir.joinpath("box_rk_nucleus.pdf"),
)

# Old

Save as Excel files

In [ ]:
# # Save to Excel with two sheets
# import pandas as pd
# sample_stats_df = pd.DataFrame(sample_stats)
# dataset_stats_df = pd.DataFrame([dataset_stats])
# output_path = f"output/{step}/graph_statistics.xlsx"
# with pd.ExcelWriter(output_path) as writer:
#     sample_stats_df.to_excel(writer, sheet_name="Sample Statistics", index=False)
#     dataset_stats_df.to_excel(writer, sheet_name="Dataset Summary", index=False)